In [1]:
# %pip install plotly
# %pip install ipywidgets

In [2]:
import cppyy
import os
import numpy as np

import planner_plot
from planner_plot import plot_trapezoid


def load_planners_and_traj():
    root = os.path.abspath(".")
    if not hasattr(cppyy.gbl, "cjp_reset"):
        cppyy.add_include_path(root)
        cppyy.add_include_path(os.path.join(root, "inc"))
        cppyy.add_include_path(os.path.join(root, "marlin files"))
        cppyy.include("constant-jerk-planner.h")
        cppyy.include("constant-jerk-planner.cpp")

    if not hasattr(cppyy.gbl, "ConstantJerkTrajectoryGenerator"):
        cppyy.add_include_path(root)
        cppyy.add_include_path(os.path.join(root, "inc"))
        cppyy.add_include_path(os.path.join(root, "marlin files"))
        cppyy.include("marlin files/trajectory_generator.h")
        cppyy.include("marlin files/trajectory_constant_jerk.h")


load_planners_and_traj()


In [3]:
# Convenience aliases matching the CJP API
cjp_reset = cppyy.gbl.cjp_reset
cjp_push_block = cppyy.gbl.cjp_push_block
cjp_recalculate = cppyy.gbl.cjp_recalculate
cjp_size = cppyy.gbl.cjp_size
cjp_get_block = cppyy.gbl.cjp_get_block
cjp_pop_front = cppyy.gbl.cjp_pop_front

CJP_BlockOut = cppyy.gbl.CJP_BlockOut


def get_block(i: int):
    out = CJP_BlockOut()
    ok = cjp_get_block(i, out)
    assert int(ok) == 1
    return {
        "mm": float(out.millimeters),
        "max_entry_speed": float(out.max_entry_speed),
        "nominal": float(out.nominal),
        "a_max": float(out.a_max),
        "j_max": float(out.j_max),
        "entry_v": float(out.entry_v),
        "entry_a": float(out.entry_a),
        "exit_v": float(out.exit_v),
        "exit_a": float(out.exit_a),
    }


def run_constant_jerk_profile(blocks, dt=0.0005):
    traj = cppyy.gbl.ConstantJerkTrajectoryGenerator()

    times = []
    positions = []
    boundaries = []

    t_offset = 0.0
    x_offset = 0.0

    for b in blocks:
        traj.reset()
        traj.plan(
            float(b["entry_v"]),
            float(b["entry_a"]),
            float(b["exit_v"]),
            float(b["exit_a"]),
            float(b["a_max"]),
            float(b["j_max"]),
            float(b["mm"]),
            float(b["nominal"]),
        )

        duration = float(traj.getTotalDuration())
        if duration <= 0.0:
            boundaries.append(t_offset)
            continue

        ts = np.linspace(0.0, duration, 2000)
        vmax = max(float(traj.getVelocityAtTime(float(t))) for t in ts)
        print("duration", duration, "nominal", b["nominal"], "entry", b["entry_v"], "exit", b["exit_v"], "true vmax", vmax)

        local_ts = []
        t_local = 0.0
        while t_local <= duration:
            local_ts.append(t_local)
            t_local += dt
        if local_ts[-1] < duration:
            local_ts.append(duration)

        local_xs = [float(traj.getDistanceAtTime(float(t))) for t in local_ts]

        times.extend((t_offset + t) for t in local_ts)
        positions.extend((x_offset + x) for x in local_xs)

        t_offset += duration
        x_offset += float(b["mm"])
        boundaries.append(t_offset)
    return np.array(times), np.array(positions), np.array(boundaries)


print("loaded OK via cppyy")


loaded OK via cppyy


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

cjp_reset()

# Base blocks: (mm, nominal, max_entry_speed, a_max, j_max)
base_blocks = [
    (35.0, 31.0, 10.0, 500.0, 8000.0),
    # (35.0, 150.0, 10.0, 500.0, 8000.0),
]

entry_slider = widgets.FloatSlider(value=0.0, min=0.0, max=200.0, step=0.5, description="entry_v", continuous_update=False)
exit_slider = widgets.FloatSlider(value=0.0, min=0.0, max=200.0, step=0.5, description="exit_v", continuous_update=False)
nominal_slider = widgets.FloatSlider(value=31.0, min=1.0, max=200.0, step=0.5, description="nominal", continuous_update=False)
max_entry_slider = widgets.FloatSlider(value=10.0, min=0.0, max=200.0, step=0.5, description="max_entry", continuous_update=False)
run_button = widgets.Button(description="Run", button_style="primary")

out = widgets.Output()


def recompute(entry_v, exit_v, nominal, max_entry_speed):
    with out:
        clear_output(wait=True)
        cjp_reset()

        total_dist_planned = 0.0
        blocks = []
        for mm, _, _, a_max, j_max in base_blocks:
            blocks.append((mm, nominal, max_entry_speed, a_max, j_max))

        for mm, nominal, max_entry_speed, a_max, j_max in blocks:
            total_dist_planned += mm
            ok = cjp_push_block(mm, max_entry_speed, nominal, a_max, j_max)
            assert int(ok) == 1

        ok = cjp_recalculate()
        print("ok?", int(ok) == 1)

        n = int(cjp_size())
        print("n blocks:", n)
        blocks_out = [get_block(i) for i in range(n)]

        if blocks_out:
            blocks_out[0]["entry_v"] = float(entry_v)
            blocks_out[0]["exit_v"] = float(exit_v)
            blocks_out[0]["nominal"] = float(nominal)

        for i, b in enumerate(blocks_out):
            print(i, b)

        t, x, boundaries = run_constant_jerk_profile(blocks_out)
        if len(t) > 1:
            mask = np.concatenate(([True], np.diff(t) > 0))
            t_plot = t[mask]
            x_plot = x[mask]
            plot_trapezoid(t_plot, x_plot, boundaries_s=boundaries)

        print("total_dist_planned:", total_dist_planned)
        if len(x) > 0:
            print("executed:", x[-1])


def on_run_clicked(_):
    recompute(entry_slider.value, exit_slider.value, nominal_slider.value, max_entry_slider.value)


run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    entry_slider,
    exit_slider,
    nominal_slider,
    max_entry_slider,
    run_button,
])

display(ui, out)
recompute(entry_slider.value, exit_slider.value, nominal_slider.value, max_entry_slider.value)
